# Phase 04H: Teacher B → S2 Sequence-KD GPU Preflight

Preflight trên 64 ảnh Train+Restval, không lưu trọng số pilot:

1. strict-load BLIP Teacher B seed 42 và S2 initialization;
2. sinh pseudo caption deterministic Beam-3;
3. tokenize lại bằng vocabulary Student;
4. kiểm tra OOV, truncation và sequence contract;
5. tính `0.5 × CE_human + 0.5 × CE_teacher_sequence`;
6. backward, gradient và Teacher frozen;
7. discarded optimizer step và atomic restore;
8. Held-out Test luôn `SEALED`.

## Kaggle inputs

1. Phase 04G compatibility entry.
2. Phase 03G Teacher B seed-42 output.
3. Phase 02A shared Student bundle.
4. Phase 01 formal output.
5. MS COCO 2014 images.
6. BLIP Base offline model package có `MODEL_SOURCE_MANIFEST.json`.

GPU T4 hoặc mạnh hơn; Internet Off.


## Cell 1 — Environment, authorization và immutable inputs


In [1]:
import copy
import csv
import hashlib
import json
import math
import os
import random
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


STARTED = time.perf_counter()
SEED = 42
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_b_s2_sequence_kd_preflight"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
DATA_DIR = OUTPUT_ROOT / "data"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, DATA_DIR, MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (name, matches)
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


if not torch.cuda.is_available():
    raise RuntimeError(
        "Phase 04H bắt buộc GPU. Chọn Kaggle GPU T4 rồi chạy lại."
    )
set_deterministic(SEED)
device = torch.device("cuda")

PHASE4G_COMPLETE_PATH = exactly_one(
    "PHASE4_TEACHER_B_S2_KD_COMPATIBILITY_ENTRY_COMPLETE.json"
)
AUTH_PATH = exactly_one(
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_PREFLIGHT_AUTHORIZED.json"
)
TEACHER_MARKER_PATH = exactly_one(
    "TEACHER_B_SEED42_FULL_RUN_COMPLETE.json"
)
PHASE2_READY_PATH = exactly_one(
    "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
PHASE1_READY_PATH = exactly_one(
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)

phase4g = read_json(PHASE4G_COMPLETE_PATH)
authorization = read_json(AUTH_PATH)
teacher_marker = read_json(TEACHER_MARKER_PATH)
phase2_ready = read_json(PHASE2_READY_PATH)
phase1_ready = read_json(PHASE1_READY_PATH)
assert phase4g["status"] == "complete"
assert phase4g["selected_method"] == "sequence_kd"
assert phase4g["direct_logit_kd"] == "prohibited"
assert phase4g["phase04h_authorized"] is True
assert authorization["status"] == "authorized"
assert authorization["experiment_id"] == "P4B_B_S2_SEQKD"
assert authorization["selected_method"] == "sequence_kd"
assert authorization["direct_logit_kd_allowed"] is False
assert teacher_marker["status"] == "complete"
assert teacher_marker["teacher_id"] == "B"
assert int(teacher_marker["seed"]) == SEED
assert teacher_marker["qualified_for_kd"] is True
assert phase2_ready["status"] == "ready"
assert phase1_ready["status"] == "ready"
for marker in [phase4g, authorization, teacher_marker]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
assert phase1_ready["test_set_status"] == "sealed"
assert phase1_ready["test_used"] is False

PHASE4G_ROOT = PHASE4G_COMPLETE_PATH.parent
policy_ref = authorization["policy"]
POLICY_PATH = resolve_inside(
    PHASE4G_ROOT, policy_ref["relative_path"]
)
assert sha256_file(POLICY_PATH) == policy_ref["sha256"]
POLICY = read_json(POLICY_PATH)
assert POLICY["status"] == "locked"
assert POLICY["supervision"]["alpha"] == 0.5
assert POLICY["supervision"]["temperature"] is None
assert POLICY["supervision"]["direct_teacher_logits_used"] is False

PHASE2_ROOT = PHASE2_READY_PATH.parent
S2_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["s2_encoder_checkpoint"],
)
DECODER_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["shared_decoder_checkpoint"],
)
DECODER_CONTRACT = resolve_inside(
    PHASE2_ROOT,
    "configs/decoder/shared_transformer_decoder_contract.json",
)
S2_ENCODER_CONTRACT = resolve_inside(
    PHASE2_ROOT,
    "configs/encoders/s2_encoder_adapter_contract.json",
)
TEACHER_ROOT = TEACHER_MARKER_PATH.parent
TEACHER_CHECKPOINT = resolve_inside(
    TEACHER_ROOT,
    teacher_marker["best_checkpoint"]["relative_path"],
)
assert sha256_file(TEACHER_CHECKPOINT) == (
    teacher_marker["best_checkpoint"]["sha256"]
)
assert sha256_file(TEACHER_CHECKPOINT) == (
    POLICY["teacher"]["checkpoint_sha256"]
)
assert sha256_file(S2_INITIAL_CHECKPOINT) == (
    POLICY["student"]["initial_encoder_sha256"]
)
assert sha256_file(DECODER_INITIAL_CHECKPOINT) == (
    POLICY["student"]["initial_decoder_sha256"]
)

model_candidates = []
for manifest in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    payload = read_json(manifest)
    if (
        payload.get("repo_id")
        == "Salesforce/blip-image-captioning-base"
        and payload.get("resolved_commit")
        == POLICY["teacher"]["revision"]
    ):
        model_candidates.append(manifest.parent.resolve())
assert len(model_candidates) == 1, model_candidates
BLIP_SOURCE_DIR = model_candidates[0]
BLIP_SOURCE_MANIFEST = read_json(
    BLIP_SOURCE_DIR / "MODEL_SOURCE_MANIFEST.json"
)
for item in BLIP_SOURCE_MANIFEST["files"]:
    path = BLIP_SOURCE_DIR / item["relative_path"]
    assert path.is_file()
    assert path.stat().st_size == int(item["size_bytes"])
    assert sha256_file(path) == item["sha256"]

write_json(
    AUDIT_DIR / "phase4h_cell01_inputs.json",
    {
        "status": "passed", "created_at_utc": now_utc(),
        "gpu": torch.cuda.get_device_name(0),
        "method": "sequence_kd", "seed": SEED,
        "teacher_checkpoint_sha256": sha256_file(TEACHER_CHECKPOINT),
        "s2_checkpoint_sha256": sha256_file(S2_INITIAL_CHECKPOINT),
        "decoder_checkpoint_sha256": sha256_file(
            DECODER_INITIAL_CHECKPOINT
        ),
        "offline_blip": "verified",
        "dataset_read": False, "model_loaded": False,
        "test_set_status": "sealed", "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - STARTED,
    },
)
print("=" * 72); print("PHASE 04H CELL 1 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Method                    : Sequence KD")
print("Offline BLIP              : VERIFIED")
print("Teacher/S2 checkpoints    : VERIFIED")
print("Dataset/model read        : False / False")
print("Test Set status           : SEALED")
print("Next                      : generate fixed pseudo captions")


PHASE 04H CELL 1 STATUS
Status                    : PASSED
GPU                       : Tesla T4
Method                    : Sequence KD
Offline BLIP              : VERIFIED
Teacher/S2 checkpoints    : VERIFIED
Dataset/model read        : False / False
Test Set status           : SEALED
Next                      : generate fixed pseudo captions


## Cell 2 — Fixed 64-image Teacher generation và Student tokenization


In [2]:
STARTED = time.perf_counter()
import pyarrow.parquet as pq
from PIL import Image
from transformers import BlipForConditionalGeneration, BlipProcessor


TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
IMAGE_MANIFEST = exactly_one("mscoco_image_manifest.parquet")
VOCAB_PATH = exactly_one("word_vocabulary_minfreq5.json")
vocabulary = read_json(VOCAB_PATH)
id_to_token = vocabulary["id_to_token"]
token_to_id = {token: index for index, token in enumerate(id_to_token)}
assert len(id_to_token) == 10080
pattern = re.compile(vocabulary["tokenizer"]["pattern"])


def student_encode(text):
    normalized = unicodedata.normalize("NFKC", text).lower()
    normalized = normalized.replace("’", "'").replace("‘", "'")
    tokens = pattern.findall(normalized)
    truncated = len(tokens) > 30
    tokens = tokens[:30]
    ids = [1] + [token_to_id.get(token, 3) for token in tokens] + [2]
    return tokens, ids, truncated


def find_leaf(name, prefix):
    matches = [
        path.resolve() for path in INPUT_ROOT.rglob(name)
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None)
    ]
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]


TRAIN_LEAF = find_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_leaf("val2014", "COCO_val2014_")
path_rows = pq.read_table(
    IMAGE_MANIFEST,
    columns=["image_id", "experiment_split", "source_partition", "image_path"],
    filters=[("experiment_split", "=", "train")],
).to_pylist()
TRAIN_PATHS = {}
for row in path_rows:
    leaf = TRAIN_LEAF if row["source_partition"] == "train2014" else VAL_LEAF
    TRAIN_PATHS[int(row["image_id"])] = leaf / Path(row["image_path"]).name
assert len(TRAIN_PATHS) == 113287

records = []
seen = set()
for batch in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=2048,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in batch.to_pylist():
        image_id = int(row["image_id"])
        if image_id in seen:
            continue
        ids = [int(value) for value in row["token_ids"]]
        assert ids[0] == 1 and ids[-1] == 2
        records.append({
            "image_id": image_id,
            "annotation_id": int(row["annotation_id"]),
            "human_ids": ids,
            "image_path": TRAIN_PATHS[image_id],
        })
        seen.add(image_id)
        if len(records) == 64:
            break
    if len(records) == 64:
        break
assert len(records) == 64

PROCESSOR = BlipProcessor.from_pretrained(
    str(BLIP_SOURCE_DIR), local_files_only=True, use_fast=False
)
TEACHER = BlipForConditionalGeneration.from_pretrained(
    str(BLIP_SOURCE_DIR), local_files_only=True
)
teacher_payload = torch.load(
    TEACHER_CHECKPOINT, map_location="cpu", weights_only=False
)
assert teacher_payload["schema_version"] == (
    "teacher_b_seed42_best_model_v1"
)
assert int(teacher_payload["seed"]) == 42
assert teacher_payload["test_set_status"] == "sealed"
assert teacher_payload["test_set_used"] is False
TEACHER.load_state_dict(teacher_payload["model"], strict=True)
del teacher_payload
TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)


def generate_for(rows):
    captions = []
    for start in range(0, len(rows), 4):
        chunk = rows[start:start + 4]
        images = []
        for row in chunk:
            with Image.open(row["image_path"]) as source:
                images.append(source.convert("RGB"))
        pixel_values = PROCESSOR(
            images=images, return_tensors="pt"
        ).pixel_values.to(device)
        with torch.inference_mode():
            token_ids = TEACHER.generate(
                pixel_values=pixel_values,
                num_beams=3,
                do_sample=False,
                length_penalty=0.7,
                min_new_tokens=2,
                max_new_tokens=31,
            )
        captions.extend(PROCESSOR.batch_decode(
            token_ids, skip_special_tokens=True
        ))
    return [caption.strip() for caption in captions]


pseudo_captions = generate_for(records)
repeat_first = generate_for(records[:4])
assert pseudo_captions[:4] == repeat_first
assert len(pseudo_captions) == 64
assert all(pseudo_captions)

pseudo_rows = []
unknown = 0
lexical = 0
truncated_count = 0
for record, caption in zip(records, pseudo_captions):
    tokens, ids, truncated = student_encode(caption)
    unknown += sum(value == 3 for value in ids[1:-1])
    lexical += len(ids) - 2
    truncated_count += int(truncated)
    record["pseudo_ids"] = ids
    pseudo_rows.append({
        "image_id": record["image_id"],
        "teacher_caption": caption,
        "student_tokens": tokens,
        "student_token_ids": ids,
        "unknown_tokens": sum(value == 3 for value in ids[1:-1]),
        "truncated": truncated,
    })
assert lexical > 0
PSEUDO_SAMPLE_PATH = DATA_DIR / "phase4h_fixed64_pseudo_captions.json"
write_json(PSEUDO_SAMPLE_PATH, {
    "schema_version": "phase4h_fixed64_pseudo_v1",
    "split": "Train+Restval",
    "teacher_seed": 42,
    "rows": pseudo_rows,
    "test_set_status": "sealed",
    "test_set_used": False,
})

write_json(AUDIT_DIR / "phase4h_cell02_pseudo_generation.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "images": 64, "unique_images": 64,
    "repeat_exact_images": 4,
    "empty_captions": 0,
    "lexical_tokens": lexical,
    "unknown_tokens": unknown,
    "unknown_rate": unknown / lexical,
    "truncated_captions": truncated_count,
    "teacher_frozen": True,
    "pseudo_split": "Train+Restval",
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72); print("PHASE 04H CELL 2 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("Pseudo captions           : 64 / 64")
print("Repeat exact              : 4 / 4")
print("Empty captions            : 0")
print("Student unknown rate      :", unknown / lexical)
print("Truncated captions        :", truncated_count)
print("Teacher frozen            : True")
print("Test Set status           : SEALED")
print("Next                      : reconstruct S2")


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: /kaggle/input/models/nguynthnhdan/blip-image-captioning-base-offline/pytorch/82a37760-pytorch/1/salesforce_blip_image_captioning_base_82a37760
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:

PHASE 04H CELL 2 STATUS
Status                    : PASSED
Pseudo captions           : 64 / 64
Repeat exact              : 4 / 4
Empty captions            : 0
Student unknown rate      : 0.0
Truncated captions        : 0
Teacher frozen            : True
Test Set status           : SEALED
Next                      : reconstruct S2


## Cell 3 — Reconstruct S2 và prepare human/pseudo targets


In [3]:
STARTED = time.perf_counter()
import timm
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from torchvision import transforms
from torchvision.transforms import InterpolationMode


decoder_config = read_json(DECODER_CONTRACT)["architecture"]
assert decoder_config["vocabulary_size"] == 10080


class S2Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = timm.create_model(
            "mobilevit_s.cvnets_in1k", pretrained=False
        )
        if hasattr(backbone, "reset_classifier"):
            backbone.reset_classifier(num_classes=0, global_pool="")
        if hasattr(backbone, "head"):
            backbone.head = nn.Identity()
        self.backbone = backbone
        self.projection = nn.Linear(640, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone.forward_features(images)
        if isinstance(feature, (tuple, list)):
            feature = [x for x in feature if torch.is_tensor(x)][-1]
        if isinstance(feature, dict):
            feature = [x for x in feature.values() if torch.is_tensor(x)][-1]
        assert feature.shape[1:] == (640, 7, 7)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(
            self.projection(tokens) + self.visual_position_embedding
        )


class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocabulary_size = int(config["vocabulary_size"])
        self.pad_id = int(config["pad_id"])
        self.d_model = int(config["d_model"])
        self.maximum_decoder_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocabulary_size, self.d_model, padding_idx=self.pad_id
        )
        self.position_embedding = nn.Embedding(
            self.maximum_decoder_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model, nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation=config["activation"], batch_first=True,
            norm_first=True, bias=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocabulary_size))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(torch.ones(
            length, length, dtype=torch.bool, device=ids.device
        ), diagonal=1)
        hidden = self.transformer_decoder(
            target, memory, tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden, self.token_embedding.weight, self.output_bias
        )


STUDENT_ENCODER = S2Encoder()
STUDENT_DECODER = SharedDecoder(decoder_config)
with torch.serialization.safe_globals([TorchVersion]):
    encoder_payload = torch.load(
        S2_INITIAL_CHECKPOINT, map_location="cpu", weights_only=True
    )
    decoder_payload = torch.load(
        DECODER_INITIAL_CHECKPOINT, map_location="cpu", weights_only=True
    )
STUDENT_ENCODER.load_state_dict(
    encoder_payload["state_dict"], strict=True
)
STUDENT_DECODER.load_state_dict(
    decoder_payload["state_dict"], strict=True
)
del encoder_payload, decoder_payload
assert sum(p.numel() for p in STUDENT_ENCODER.parameters()) == 5114784
assert sum(p.numel() for p in STUDENT_DECODER.parameters()) == 5759328
INITIAL_ENCODER = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
INITIAL_DECODER = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)

student_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])


def pad_full(sequences):
    maximum = max(len(x) for x in sequences)
    result = torch.zeros(len(sequences), maximum, dtype=torch.long)
    for index, sequence in enumerate(sequences):
        result[index, :len(sequence)] = torch.tensor(sequence)
    return result


smoke_records = records[:8]
images = []
for row in smoke_records:
    with Image.open(row["image_path"]) as source:
        images.append(student_transform(source.convert("RGB")))
STUDENT_IMAGES = torch.stack(images).to(device)
HUMAN_FULL = pad_full([row["human_ids"] for row in smoke_records]).to(device)
PSEUDO_FULL = pad_full([row["pseudo_ids"] for row in smoke_records]).to(device)
assert HUMAN_FULL.shape[1] <= 32
assert PSEUDO_FULL.shape[1] <= 32


def targets(full):
    return full[:, :-1], full[:, 1:], full[:, :-1].ne(0)


HUMAN_INPUT, HUMAN_LABELS, HUMAN_MASK = targets(HUMAN_FULL)
PSEUDO_INPUT, PSEUDO_LABELS, PSEUDO_MASK = targets(PSEUDO_FULL)
write_json(AUDIT_DIR / "phase4h_cell03_student_model.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "student_parameters": 10874112,
    "student_images": list(STUDENT_IMAGES.shape),
    "human_decoder_shape": list(HUMAN_INPUT.shape),
    "pseudo_decoder_shape": list(PSEUDO_INPUT.shape),
    "strict_checkpoint_loads": "2 / 2",
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72); print("PHASE 04H CELL 3 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("S2 parameters             : 10874112")
print("Strict checkpoint loads   : 2 / 2")
print("Images                    :", list(STUDENT_IMAGES.shape))
print("Human target              :", list(HUMAN_LABELS.shape))
print("Teacher target            :", list(PSEUDO_LABELS.shape))
print("Test Set status           : SEALED")
print("Next                      : combined loss and backward")


PHASE 04H CELL 3 STATUS
Status                    : PASSED
S2 parameters             : 10874112
Strict checkpoint loads   : 2 / 2
Images                    : [8, 3, 224, 224]
Human target              : [8, 13]
Teacher target            : [8, 12]
Test Set status           : SEALED
Next                      : combined loss and backward


## Cell 4 — Combined Sequence-KD loss và gradient


In [4]:
STARTED = time.perf_counter()


def set_student_train():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    count = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            count += 1
    assert count == 32
    return count


def sequence_ce(logits, labels):
    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        labels.reshape(-1),
        ignore_index=0,
    )


set_student_train()
STUDENT_ENCODER.zero_grad(set_to_none=True)
STUDENT_DECODER.zero_grad(set_to_none=True)
memory = STUDENT_ENCODER(STUDENT_IMAGES)
human_logits = STUDENT_DECODER(HUMAN_INPUT, memory, HUMAN_MASK)
pseudo_logits = STUDENT_DECODER(PSEUDO_INPUT, memory, PSEUDO_MASK)
human_ce = sequence_ce(human_logits, HUMAN_LABELS)
pseudo_ce = sequence_ce(pseudo_logits, PSEUDO_LABELS)
combined_loss = 0.5 * human_ce + 0.5 * pseudo_ce
assert torch.isfinite(combined_loss)
combined_loss.backward()
student_gradients = [
    parameter.grad for module in [STUDENT_ENCODER, STUDENT_DECODER]
    for parameter in module.parameters() if parameter.requires_grad
]
assert student_gradients
assert all(gradient is not None for gradient in student_gradients)
assert all(torch.isfinite(gradient).all() for gradient in student_gradients)
gradient_norm = torch.sqrt(sum(
    gradient.detach().float().pow(2).sum()
    for gradient in student_gradients
))
assert float(gradient_norm) > 0.0
assert all(parameter.grad is None for parameter in TEACHER.parameters())

write_json(AUDIT_DIR / "phase4h_cell04_loss_backward.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "human_ce": float(human_ce.detach()),
    "teacher_sequence_ce": float(pseudo_ce.detach()),
    "combined_loss": float(combined_loss.detach()),
    "alpha": 0.5,
    "temperature": None,
    "gradient_l2": float(gradient_norm),
    "student_gradients_finite": True,
    "teacher_gradients": "none",
    "direct_teacher_logits_used": False,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72); print("PHASE 04H CELL 4 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("Human CE                  :", float(human_ce.detach()))
print("Teacher-sequence CE       :", float(pseudo_ce.detach()))
print("Combined loss             :", float(combined_loss.detach()))
print("Student gradient L2       :", float(gradient_norm))
print("Teacher gradients         : NONE")
print("Direct teacher logits     : False")
print("Test Set status           : SEALED")
print("Next                      : discarded optimizer/resume pilot")


PHASE 04H CELL 4 STATUS
Status                    : PASSED
Human CE                  : 11.00248908996582
Teacher-sequence CE       : 11.510306358337402
Combined loss             : 11.256397247314453
Student gradient L2       : 10.499606132507324
Teacher gradients         : NONE
Direct teacher logits     : False
Test Set status           : SEALED
Next                      : discarded optimizer/resume pilot


## Cell 5 — Discarded optimizer step và atomic restore


In [5]:
STARTED = time.perf_counter()
TEMP_CHECKPOINT = OUTPUT_ROOT / "phase4h_temporary_preflight.pt"


def model_state_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def state_exact(left, right):
    return left.keys() == right.keys() and all(
        torch.equal(left[key], right[key]) for key in left
    )


def cpu_tree(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [cpu_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_tree(item) for item in value)
    return copy.deepcopy(value)


def nested_exact(left, right):
    if torch.is_tensor(left) or torch.is_tensor(right):
        return (
            torch.is_tensor(left)
            and torch.is_tensor(right)
            and torch.equal(left.detach().cpu(), right.detach().cpu())
        )
    if isinstance(left, dict) or isinstance(right, dict):
        return (
            isinstance(left, dict)
            and isinstance(right, dict)
            and left.keys() == right.keys()
            and all(nested_exact(left[key], right[key]) for key in left)
        )
    if isinstance(left, (list, tuple)) or isinstance(right, (list, tuple)):
        return (
            isinstance(left, type(right))
            and len(left) == len(right)
            and all(nested_exact(a, b) for a, b in zip(left, right))
        )
    return left == right


STUDENT_ENCODER.load_state_dict(INITIAL_ENCODER, strict=True)
STUDENT_DECODER.load_state_dict(INITIAL_DECODER, strict=True)
backbone = list(STUDENT_ENCODER.backbone.parameters())
adapter = [
    *STUDENT_ENCODER.projection.parameters(),
    STUDENT_ENCODER.visual_position_embedding,
    *STUDENT_ENCODER.output_norm.parameters(),
]
optimizer = torch.optim.AdamW([
    {"params": backbone, "lr": 1e-4, "name": "backbone"},
    {"params": adapter, "lr": 3e-4, "name": "adapter"},
    {"params": list(STUDENT_DECODER.parameters()), "lr": 3e-4, "name": "decoder"},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda step: min(1.0, (step + 1) / 1771)
)
scaler = torch.amp.GradScaler("cuda", enabled=True, init_scale=1024.0)
optimizer.zero_grad(set_to_none=True)
set_student_train()
with torch.amp.autocast("cuda", dtype=torch.float16):
    memory = STUDENT_ENCODER(STUDENT_IMAGES)
    h = STUDENT_DECODER(HUMAN_INPUT, memory, HUMAN_MASK)
    p = STUDENT_DECODER(PSEUDO_INPUT, memory, PSEUDO_MASK)
    pilot_loss = 0.5 * sequence_ce(h, HUMAN_LABELS) + 0.5 * sequence_ce(
        p, PSEUDO_LABELS
    )
scaler.scale(pilot_loss).backward()
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(
    list(STUDENT_ENCODER.parameters())
    + list(STUDENT_DECODER.parameters()), 1.0
)
scale_before = scaler.get_scale()
scaler.step(optimizer)
scaler.update()
assert scaler.get_scale() >= scale_before
scheduler.step()
post_encoder = model_state_cpu(STUDENT_ENCODER)
post_decoder = model_state_cpu(STUDENT_DECODER)
assert not state_exact(post_encoder, INITIAL_ENCODER)
assert not state_exact(post_decoder, INITIAL_DECODER)

expected_optimizer = cpu_tree(optimizer.state_dict())
expected_scheduler = cpu_tree(scheduler.state_dict())
expected_scaler = cpu_tree(scaler.state_dict())
payload = {
    "schema_version": "phase4h_preflight_checkpoint_v1",
    "seed": 42,
    "encoder": post_encoder,
    "decoder": post_decoder,
    "optimizer": expected_optimizer,
    "scheduler": expected_scheduler,
    "scaler": expected_scaler,
    "cpu_rng": torch.get_rng_state(),
    "cuda_rng": torch.cuda.get_rng_state_all(),
    "test_set_status": "sealed",
    "test_set_used": False,
}
torch.save(payload, TEMP_CHECKPOINT)
loaded = torch.load(TEMP_CHECKPOINT, map_location="cpu", weights_only=False)
assert state_exact(loaded["encoder"], post_encoder)
assert state_exact(loaded["decoder"], post_decoder)
assert nested_exact(loaded["optimizer"], expected_optimizer)
assert nested_exact(loaded["scheduler"], expected_scheduler)
assert nested_exact(loaded["scaler"], expected_scaler)
assert torch.equal(loaded["cpu_rng"], torch.get_rng_state())
assert all(
    torch.equal(a, b)
    for a, b in zip(loaded["cuda_rng"], torch.cuda.get_rng_state_all())
)
del loaded

STUDENT_ENCODER.load_state_dict(INITIAL_ENCODER, strict=True)
STUDENT_DECODER.load_state_dict(INITIAL_DECODER, strict=True)
assert state_exact(model_state_cpu(STUDENT_ENCODER), INITIAL_ENCODER)
assert state_exact(model_state_cpu(STUDENT_DECODER), INITIAL_DECODER)
TEMP_CHECKPOINT.unlink()
assert not TEMP_CHECKPOINT.exists()

write_json(AUDIT_DIR / "phase4h_cell05_discarded_pilot.json", {
    "status": "passed", "created_at_utc": now_utc(),
    "optimizer_steps": 1, "weights_updated": True,
    "checkpoint_restore_exact": True,
    "optimizer_scheduler_scaler_restored": True,
    "rng_restored_exact": True,
    "initial_state_restored_exact": True,
    "temporary_checkpoint_deleted": True,
    "training_started": False,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72); print("PHASE 04H CELL 5 STATUS"); print("=" * 72)
print("Status                    : PASSED")
print("Discarded optimizer steps : 1")
print("Parameter update observed : True")
print("Atomic restore exact      : True")
print("Initial restore exact     : True")
print("Temporary checkpoint      : DELETED")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Next                      : package preflight")


PHASE 04H CELL 5 STATUS
Status                    : PASSED
Discarded optimizer steps : 1
Parameter update observed : True
Atomic restore exact      : True
Initial restore exact     : True
Temporary checkpoint      : DELETED
Full training started     : False
Test Set status           : SEALED
Next                      : package preflight


## Cell 6 — Formal package và pseudo-corpus authorization


In [6]:
STARTED = time.perf_counter()
NEXT_AUTH = OUTPUT_ROOT / (
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_PSEUDO_CORPUS_AUTHORIZED.json"
)
write_json(NEXT_AUTH, {
    "marker_version": (
        "phase4_teacher_b_s2_sequence_kd_pseudo_corpus_authorized_v1"
    ),
    "created_at_utc": now_utc(), "status": "authorized",
    "experiment_id": "P4B_B_S2_SEQKD",
    "teacher_seed": 42, "selected_method": "sequence_kd",
    "required_images": 113287,
    "pseudo_caption_split": "Train+Restval only",
    "generation": POLICY["teacher"]["generation"],
    "resume_and_sharding_required": True,
    "next_phase": "Phase 04I full pseudo-caption corpus generation",
    "test_set_status": "sealed", "test_set_used": False,
})

MANIFEST_CSV = MANIFEST_DIR / "phase4h_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase4h_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase4h_SHA256SUMS.txt"
FINAL_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_PREFLIGHT_COMPLETE.json"
)
artifacts = sorted([
    path for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
    and path not in {MANIFEST_CSV, MANIFEST_JSON, SHA256SUMS, FINAL_MARKER}
    and not path.name.endswith(".tmp")
], key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix())
rows = [{
    "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
    "size_bytes": path.stat().st_size,
    "sha256": sha256_file(path),
} for path in artifacts]
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader(); writer.writerows(rows)
write_json(MANIFEST_JSON, {
    "schema_version": "phase4h_manifest_v1",
    "artifact_count": len(rows), "artifacts": rows,
})
SHA256SUMS.write_text("".join(
    f'{row["sha256"]}  {row["relative_path"]}\n' for row in rows
), encoding="utf-8")
for row in rows:
    path = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert path.stat().st_size == int(row["size_bytes"])
    assert sha256_file(path) == row["sha256"]

write_json(FINAL_MARKER, {
    "marker_version": (
        "phase4_teacher_b_s2_sequence_kd_preflight_complete_v1"
    ),
    "created_at_utc": now_utc(), "status": "complete",
    "experiment_id": "P4B_B_S2_SEQKD",
    "teacher": "Teacher B — BLIP Base",
    "student": "S2 — MobileViT-S",
    "method": "sequence_kd", "seed": 42,
    "fixed_images": 64,
    "teacher_generation_repeatability": "passed",
    "student_retokenization": "passed",
    "combined_loss_backward": "passed",
    "discarded_optimizer_pilot": "passed",
    "atomic_restore": "passed",
    "initial_state_restored": True,
    "training_started": False,
    "phase04i_authorized": True,
    "next_authorization": {
        "relative_path": NEXT_AUTH.relative_to(OUTPUT_ROOT).as_posix(),
        "sha256": sha256_file(NEXT_AUTH),
    },
    "manifest": {
        "csv": {"relative_path": MANIFEST_CSV.relative_to(OUTPUT_ROOT).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV)},
        "json": {"relative_path": MANIFEST_JSON.relative_to(OUTPUT_ROOT).as_posix(),
                 "sha256": sha256_file(MANIFEST_JSON)},
        "sha256sums": {"relative_path": SHA256SUMS.relative_to(OUTPUT_ROOT).as_posix(),
                       "sha256": sha256_file(SHA256SUMS)},
    },
    "manifest_rows": len(rows), "manifest_verification": "all_passed",
    "test_set_status": "sealed", "test_set_used": False,
})
print("=" * 72); print("PHASE 04H CELL 6 STATUS"); print("=" * 72)
print("Status                    : COMPLETE")
print("Teacher generation        : PASSED")
print("Student re-tokenization   : PASSED")
print("Combined loss/backward    : PASSED")
print("Discarded optimizer pilot : PASSED")
print("Atomic restore            : PASSED")
print("Manifest rows             :", len(rows))
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Phase 04I pseudo corpus   : AUTHORIZED")
print("Marker                    :", FINAL_MARKER)


PHASE 04H CELL 6 STATUS
Status                    : COMPLETE
Teacher generation        : PASSED
Student re-tokenization   : PASSED
Combined loss/backward    : PASSED
Discarded optimizer pilot : PASSED
Atomic restore            : PASSED
Manifest rows             : 7
Full training started     : False
Test Set status           : SEALED
Phase 04I pseudo corpus   : AUTHORIZED
Marker                    : /kaggle/working//phase4_teacher_b_s2_sequence_kd_preflight/PHASE4_TEACHER_B_S2_SEQUENCE_KD_PREFLIGHT_COMPLETE.json
